# Sala Chaturamuk Phaichit — Gaussian Splatting, 352 photographs (resumable)

Every expensive stage is written to Drive the moment it finishes, and every cell checks Drive
before doing any work. So if the runtime dies you start a fresh one, **Run all**, and it skips
straight to wherever it got to.

| Stage | Cached in Drive as | Recompute cost if lost |
|---|---|---|
| Feature extraction | `db_extracted.db` | 25 min |
| Matching | `db_matched.db` | 45–60 min |
| Mapper | `sparse/` (+ snapshots) | 15–30 min |
| Training | `chkpnt*.pth` every 5,000 iters | up to 40 min |

Undistortion and sky masking are *not* cached — they take about 15 minutes and cost more to copy
to Drive than to redo.

**Runtime → Change runtime type → T4 GPU.** Not the P100: it compiles the CUDA extensions and
then fails an hour into training.

## 1 · GPU check

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
name = torch.cuda.get_device_name(0); major, minor = torch.cuda.get_device_capability(0)
print(f'device: {name}  (compute {major}.{minor})  torch {torch.__version__}')
assert major >= 7, f'{name} is compute {major}.{minor}; the P100 fails during training. Use a T4.'
print('OK')

## 2 · Drive, photographs, and where we left off

Prints a resume report so you can see what is already banked before anything runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob, shutil, time

hits = glob.glob('/content/drive/MyDrive/**/sala_352.zip', recursive=True)
assert hits, 'sala_352.zip not found anywhere in Drive - has the upload finished?'
ZIP  = hits[0]
OUT  = os.path.join(os.path.dirname(ZIP), 'sala_splat_out')
WORK = '/content/work'
SCENE, SRC = f'{WORK}/scene', f'{WORK}/scene/input'
DB   = f'{SCENE}/database.db'
MODEL_DIR = f'{OUT}/sala_352_masked'
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT, exist_ok=True); os.makedirs(SCENE, exist_ok=True)

DB_EXTRACTED = f'{OUT}/db_extracted.db'
DB_MATCHED   = f'{OUT}/db_matched.db'
SPARSE_DRIVE = f'{OUT}/sparse'

def have(p): return os.path.exists(p)
print('zip     :', ZIP)
print('outputs :', OUT)
print()
print('--- resume report ---')
print(f'  extraction cached : {have(DB_EXTRACTED)}')
print(f'  matching cached   : {have(DB_MATCHED)}')
print(f'  mapper cached     : {have(SPARSE_DRIVE)}')
ck = sorted(glob.glob(f'{MODEL_DIR}/chkpnt*.pth'),
            key=lambda p: int(''.join(c for c in os.path.basename(p) if c.isdigit())))
print(f'  training ckpts    : {[os.path.basename(c) for c in ck] if ck else "none"}')

with zipfile.ZipFile(ZIP) as z: z.extractall(f'{WORK}/raw')
imgs = sorted(glob.glob(f'{WORK}/raw/**/*.jpg', recursive=True))
print(f'\n{len(imgs)} photographs unpacked')
assert len(imgs) > 300

## 3 · Downscale to 1600 px

In [ ]:
import cv2
os.makedirs(SRC, exist_ok=True)
if len(glob.glob(f'{SRC}/*.jpg')) >= 300:
    print('already downscaled, skipping')
else:
    def tag(p):
        low = p.lower()
        return 'a_v1' if 'version1' in low else ('b_v2' if 'version2' in low else 'c_xx')
    groups = {}
    for p in imgs: groups.setdefault(tag(p), []).append(p)
    for g in groups: groups[g].sort()
    for g, ps in sorted(groups.items()):
        for k, p in enumerate(ps):
            im = cv2.imread(p)
            if im is None: continue
            h, w = im.shape[:2]
            if w > 1600: im = cv2.resize(im, (1600, round(h*1600/w)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(f'{SRC}/{g}_{k:04d}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])
        print(f'{g}: {len(ps)}')
print('images ready:', len(glob.glob(f'{SRC}/*.jpg')))

## 4 · Install COLMAP

In [ ]:
import os
# Colab is headless. COLMAP links Qt, whose default plugin wants an X display,
# so without this every colmap call aborts before doing any work.
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
if shutil.which('colmap') is None:
    !apt-get -qq update && apt-get -qq install -y colmap > /dev/null
print(shutil.which('colmap'))

## 5 · Feature extraction — cached

Saved to Drive as soon as it finishes. GPU SIFT needs an OpenGL context, which a headless
runtime cannot give, so this runs on CPU.

In [ ]:
if have(DB_MATCHED):
    shutil.copy(DB_MATCHED, DB); print('matched database restored - extraction and matching both skipped')
elif have(DB_EXTRACTED):
    shutil.copy(DB_EXTRACTED, DB); print('extracted database restored from Drive - skipping extraction')
else:
    !rm -f {DB}
    t0 = time.time()
    !QT_QPA_PLATFORM=offscreen colmap feature_extractor \
        --database_path {DB} --image_path {SRC} \
        --ImageReader.single_camera 1 \
        --ImageReader.camera_model SIMPLE_RADIAL \
        --SiftExtraction.use_gpu 0 \
        --SiftExtraction.max_image_size 1600 \
        --SiftExtraction.max_num_features 8192
    assert have(DB), 'extraction produced no database'
    shutil.copy(DB, DB_EXTRACTED)
    print(f'extraction done in {(time.time()-t0)/60:.1f} min, banked to Drive')

## 6 · Matching — cached

The pavilion is four-faced, so views 90 degrees apart look alike; exhaustive matching pairs up
different faces and the mapper folds the geometry. Sequential matching avoids that, and loop
detection bridges the join between the two separate walks.

In [ ]:
if have(DB_MATCHED):
    print('matching already cached - skipping')
else:
    VOCAB = '/content/vocab_tree_flickr100K_words32K.bin'
    if not os.path.exists(VOCAB):
        !wget -q https://demuc.de/colmap/vocab_tree_flickr100K_words32K.bin -O {VOCAB}
    t0 = time.time()
    !QT_QPA_PLATFORM=offscreen colmap sequential_matcher \
        --database_path {DB} \
        --SiftMatching.use_gpu 0 \
        --SiftMatching.max_num_matches 16384 \
        --SequentialMatching.overlap 8 \
        --SequentialMatching.loop_detection 1 \
        --SequentialMatching.vocab_tree_path {VOCAB}
    shutil.copy(DB, DB_MATCHED)
    print(f'matching done in {(time.time()-t0)/60:.1f} min, banked to Drive')

## 7 · Mapper — cached, with snapshots

`snapshot_images_freq` writes a partial reconstruction every 40 registered images, so even a
mapper that dies half way leaves something to inspect rather than nothing.

In [ ]:
SPARSE = f'{SCENE}/sparse'
if have(SPARSE_DRIVE) and glob.glob(f'{SPARSE_DRIVE}/*/images.bin') + glob.glob(f'{SPARSE_DRIVE}/*/images.txt'):
    if os.path.exists(SPARSE): shutil.rmtree(SPARSE)
    shutil.copytree(SPARSE_DRIVE, SPARSE); print('mapper result restored from Drive - skipping')
else:
    os.makedirs(SPARSE, exist_ok=True)
    SNAP = f'{OUT}/mapper_snapshots'; os.makedirs(SNAP, exist_ok=True)
    t0 = time.time()
    !QT_QPA_PLATFORM=offscreen colmap mapper \
        --database_path {DB} --image_path {SRC} --output_path {SPARSE} \
        --Mapper.snapshot_path {SNAP} \
        --Mapper.snapshot_images_freq 40
    assert glob.glob(f'{SPARSE}/*'), 'mapper produced no model'
    if os.path.exists(SPARSE_DRIVE): shutil.rmtree(SPARSE_DRIVE)
    shutil.copytree(SPARSE, SPARSE_DRIVE)
    print(f'mapper done in {(time.time()-t0)/60:.1f} min, banked to Drive')
!ls {SPARSE}

## 8 · Fold check — read this before training

A walk photographed in order should have consecutive cameras close together. If the geometry has
folded, some consecutive pairs land far apart, and the ratio of largest to median step exposes it
regardless of scale.

Reference from the previous failure: **7.0** for capture 2 alone (healthy) versus **36.0** for the
same photographs inside the folded combined model, and **102.2** for capture 1.

**This cell raises an error if it detects a fold**, so a `Run all` stops here instead of spending
three hours training on bad camera poses.

In [ ]:
import numpy as np
sub = sorted(glob.glob(f'{SPARSE}/*'))
print('sub-models:', [os.path.basename(s) for s in sub])
MODEL = sub[0]
if not os.path.exists(f'{MODEL}/images.txt'):
    !QT_QPA_PLATFORM=offscreen colmap model_converter --input_path {MODEL} --output_path {MODEL} --output_type TXT

pos = {}
for line in open(f'{MODEL}/images.txt'):
    if line.startswith('#') or not line.strip(): continue
    f = line.split()
    if len(f) < 10 or not f[0].isdigit(): continue
    qw,qx,qy,qz,tx,ty,tz = map(float, f[1:8]); nm = f[9]
    q = np.array([qw,qx,qy,qz]); q /= np.linalg.norm(q); w,x,y,z = q
    R = np.array([[1-2*(y*y+z*z), 2*(x*y-w*z), 2*(x*z+w*y)],
                  [2*(x*y+w*z), 1-2*(x*x+z*z), 2*(y*z-w*x)],
                  [2*(x*z-w*y), 2*(y*z+w*x), 1-2*(x*x+y*y)]])
    pos[nm] = -R.T @ np.array([tx,ty,tz])
print(f'registered {len(pos)} of {len(os.listdir(SRC))} images')

folded = []
for pre in ('a_v1','b_v2'):
    ns = sorted(n for n in pos if n.startswith(pre))
    if len(ns) < 5: continue
    P = np.array([pos[n] for n in ns])
    step = np.linalg.norm(np.diff(P, axis=0), axis=1)
    ratio = step.max()/np.median(step)
    ok = ratio < 15
    print(f'{pre}: {len(ns)} imgs  max/median step = {ratio:6.1f}  '
          f'jumps over 5x = {(step > 5*np.median(step)).sum():3d}  '
          f'{"LOOKS OK" if ok else "*** POSSIBLE FOLD ***"}')
    if not ok: folded.append(pre)

assert not folded, (f'Fold suspected in {folded}. Do NOT train on these poses. '
                    'Re-run matching with --SequentialMatching.loop_detection 0, or '
                    'reconstruct each capture separately.')
print('\nfold check passed')

## 9 · Undistort to a pinhole camera

In [ ]:
PIN = f'{WORK}/scene_pinhole'
if os.path.exists(f'{PIN}/images') and len(glob.glob(f'{PIN}/images/*')) > 300:
    print('already undistorted, skipping')
else:
    !QT_QPA_PLATFORM=offscreen colmap image_undistorter --image_path {SRC} --input_path {MODEL} \
        --output_path {PIN} --output_type COLMAP --max_image_size 1600
print(len(glob.glob(f'{PIN}/images/*')), 'undistorted images')

## 10 · Remove the sky

In the existing 15,000-iteration model, 196,775 of 554,618 Gaussians (35.5%) are bright and
semi-transparent — sky floaters — and the largest 1% by scale sit at a median radius of 9.92
against a scene median of 2.30. Over a third of the model's capacity is painting haze.

The sky is painted **black** rather than carried as alpha. 3DGS renders against black by default,
so a black region costs nothing to explain and the optimiser leaves it empty.

Check the contact sheet. A leftover sliver of sky is harmless; a chewed roofline is not — raise
`V_MIN` if edges are being eaten.

In [ ]:
import numpy as np, cv2
V_MIN, S_MAX, ERODE = 140, 50, 9
MARK = f'{PIN}/.sky_removed'

def sky_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV); S, V = hsv[...,1], hsv[...,2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    grad = cv2.magnitude(cv2.Sobel(gray, cv2.CV_32F,1,0,ksize=3),
                         cv2.Sobel(gray, cv2.CV_32F,0,1,ksize=3))
    smooth = cv2.blur((grad < 25).astype(np.uint8), (9,9)) > 0.7
    cand = ((S < S_MAX) & (V > V_MIN) & smooth).astype(np.uint8)
    cand = cv2.morphologyEx(cand, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))
    _, lab = cv2.connectedComponents(cand)
    top = set(np.unique(lab[: img.shape[0]//20])) - {0}
    return cv2.erode(np.isin(lab, list(top)).astype(np.uint8), np.ones((ERODE,ERODE), np.uint8))

files = sorted(glob.glob(f'{PIN}/images/*'))
if os.path.exists(MARK):
    print('sky already removed, skipping')
else:
    frac = []
    for p in files:
        im = cv2.imread(p); m = sky_mask(im); im[m.astype(bool)] = 0
        cv2.imwrite(p, im, [cv2.IMWRITE_JPEG_QUALITY, 95]); frac.append(m.mean())
    open(MARK,'w').write('done')
    frac = np.array(frac)
    print(f'sky removed: mean {frac.mean()*100:.1f}% per frame '
          f'(min {frac.min()*100:.1f}%, max {frac.max()*100:.1f}%)')

import matplotlib.pyplot as plt
sel = files[::max(1,len(files)//8)][:8]
fig, ax = plt.subplots(2,4, figsize=(16,6))
for a,p in zip(ax.ravel(), sel):
    a.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); a.axis('off')
    a.set_title(os.path.basename(p), fontsize=8)
plt.tight_layout(); plt.show()

## 11 · Install 3D Gaussian Splatting

In [ ]:
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'      # T4; without this the build can guess wrong
if not os.path.exists('/content/gaussian-splatting'):
    %cd /content
    !git clone --recursive -q https://github.com/graphdeco-inria/gaussian-splatting
    %cd /content/gaussian-splatting
    !pip install -q plyfile
    !pip install -q submodules/diff-gaussian-rasterization
    !pip install -q submodules/simple-knn
else:
    print('already installed')

## 12 · Train — resumable

`--checkpoint_iterations` writes a full optimiser state every 5,000 iterations, and the cell
automatically restarts from the newest one with `--start_checkpoint`. So a dead runtime costs at
most 5,000 iterations, not the whole run.

`--save_iterations` additionally writes viewable `.ply` models at 7k / 15k / 30k.

`--eval` holds out every 8th photograph, so the scores in the next cell are measured against
images the model never saw.

In [ ]:
%cd /content/gaussian-splatting
os.makedirs(MODEL_DIR, exist_ok=True)

ck = sorted(glob.glob(f'{MODEL_DIR}/chkpnt*.pth'),
            key=lambda p: int(''.join(c for c in os.path.basename(p) if c.isdigit())))
RESUME = ck[-1] if ck else None
print('resuming from', RESUME if RESUME else 'scratch')

CKPTS = '5000 10000 15000 20000 25000 30000'
if RESUME:
    !python train.py -s {PIN} -m {MODEL_DIR} --iterations 30000 \
        --save_iterations 7000 15000 30000 \
        --checkpoint_iterations {CKPTS} \
        --start_checkpoint {RESUME} --eval
else:
    !python train.py -s {PIN} -m {MODEL_DIR} --iterations 30000 \
        --save_iterations 7000 15000 30000 \
        --checkpoint_iterations {CKPTS} --eval

## 13 · Render the held-out views and score them

In [ ]:
%cd /content/gaussian-splatting
!python render.py  -m {MODEL_DIR}
!python metrics.py -m {MODEL_DIR}
import json
print(json.dumps(json.load(open(f'{MODEL_DIR}/results.json')), indent=2))
print('\nprevious run, 146 images, no masks:  PSNR 22.09  SSIM 0.785  LPIPS 0.346')

## 14 · Crop the obstacles

Sky masking removes the haze. The trees, lawn and footpath are genuinely photographed and so are
genuinely reconstructed — they come off by cropping in space, the same cut used on the
photogrammetry mesh. Adjust `KEEP_R` from the printed percentiles.

In [ ]:
import numpy as np
from plyfile import PlyData, PlyElement
SRC_PLY = f'{MODEL_DIR}/point_cloud/iteration_30000/point_cloud.ply'
if not os.path.exists(SRC_PLY):
    cands = sorted(glob.glob(f'{MODEL_DIR}/point_cloud/iteration_*/point_cloud.ply'),
                   key=lambda p: int(p.split('iteration_')[1].split('/')[0]))
    SRC_PLY = cands[-1]
print('using', SRC_PLY)

ply = PlyData.read(SRC_PLY); v = ply['vertex']
xyz = np.stack([v['x'], v['y'], v['z']], 1).astype(np.float64)
C = np.array(list(pos.values()))
centre = np.median(C, axis=0); ring = np.median(np.linalg.norm(C - centre, axis=1))
r = np.linalg.norm(xyz[:, [0,2]] - centre[[0,2]], axis=1)
h = xyz[:,1] - np.median(xyz[:,1])
print(f'{len(xyz):,} gaussians   camera ring radius {ring:.2f}')
for q in (50,70,80,90,95,99): print(f'  radius p{q:<3d} = {np.percentile(r,q):6.2f}')

KEEP_R, KEEP_H = 0.55*ring, 2.5*ring
keep = (r < KEEP_R) & (np.abs(h) < KEEP_H)
print(f'\nkeeping {keep.sum():,} of {len(xyz):,} ({keep.mean()*100:.1f}%) at KEEP_R={KEEP_R:.2f}')

DST = f'{OUT}/sala_352_masked_cropped.ply'
PlyData([PlyElement.describe(v.data[keep], 'vertex')]).write(DST)
print('wrote', DST, f'({os.path.getsize(DST)/1e6:.1f} MB)')